# Lecture 15: Dialogue & Chatbot Systems
### NLP Course 2027

---

## Learning Outcomes
- Understand the taxonomy of dialogue systems
- Build rule-based, retrieval-based, and generative chatbots
- Design NLU components: intent classification and slot filling
- Prototype a multi-turn dialogue manager

**Primary Reference:** *Practical NLP* Ch.9

## 1. Types of Dialogue Systems

```
Dialogue Systems
|-- Task-Oriented (TOD)
|   |-- Goal: complete a specific task
|   |-- Examples: booking flights, customer support
|   +-- Components: NLU -> Dialogue Manager -> NLG
|
+-- Open-Domain (Chit-chat)
    |-- Goal: engaging conversation
    |-- Examples: ChatGPT, DialoGPT
    +-- Approaches: retrieval-based, generative
```

### Task-Oriented Pipeline
```
  User: 'Book a flight to Paris on Friday'
         |
  NLU:  intent=book_flight, slots={destination:Paris, date:Friday}
         |
  DM:   state_tracker -> decide action -> ask/confirm/execute
         |
  NLG:  'Sure! What time would you like to depart for Paris?'
```

In [1]:
import random

class RuleBasedChatbot:
    """Simple rule-based chatbot using keyword matching."""
    RULES = [
        ({'hello','hi','hey','greetings'},
         ['Hello! How can I help?', 'Hi there! What can I do for you?']),
        ({'bye','goodbye','exit','quit'},
         ['Goodbye! Have a great day!', 'See you later!']),
        ({'help','support','problem','issue'},
         ["I'm here to help. Please describe your issue."]),
        ({'hours','open','schedule'},
         ['We are open Monday-Friday, 9am-6pm Eastern Time.']),
        ({'price','cost','fee','payment'},
         ['Pricing depends on your plan. See our website for details.']),
        ({'name','who are you'},
         ["I'm NLPBot, your automated assistant!"]),
    ]
    FALLBACK = "I'm not sure about that. Could you rephrase?"

    def respond(self, user_input):
        words = set(user_input.lower().split())
        for keywords, responses in self.RULES:
            if words & keywords:
                return random.choice(responses)
        return self.FALLBACK

bot = RuleBasedChatbot()
for msg in ['Hello!', 'What are your hours?', 'I need help', 'How much does it cost?', 'Bye!']:
    print(f'User:  {msg}')
    print(f'Bot:   {bot.respond(msg)}')
    print()

User:  Hello!
Bot:   I'm not sure about that. Could you rephrase?

User:  What are your hours?
Bot:   I'm not sure about that. Could you rephrase?

User:  I need help
Bot:   I'm here to help. Please describe your issue.

User:  How much does it cost?
Bot:   I'm not sure about that. Could you rephrase?

User:  Bye!
Bot:   I'm not sure about that. Could you rephrase?



## 2. Intent Classification

ML-based intent detection generalizes far better than keyword rules.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

train_data = [
    ('Hello there!', 'greeting'), ('Hi, how are you?', 'greeting'),
    ('Good morning', 'greeting'), ('Hey!', 'greeting'),
    ('Where is my order?', 'check_order'), ('Track my package', 'check_order'),
    ('Status of my order?', 'check_order'), ('Has my shipment shipped?', 'check_order'),
    ('I want to cancel my order', 'cancel_order'),
    ('Please cancel my purchase', 'cancel_order'),
    ('How do I cancel?', 'cancel_order'),
    ('I want to return this item', 'return_item'),
    ('How do I return a product?', 'return_item'),
    ('Can I get a refund?', 'return_item'),
    ('Goodbye!', 'farewell'), ('See you later', 'farewell'),
    ('Thanks, bye!', 'farewell'), ('That will be all', 'farewell'),
]

utterances, intents = zip(*train_data)
intent_clf = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=500)),
])
intent_clf.fit(utterances, intents)

test_msgs = [
    'Hi there!', 'When will my package arrive?',
    'I need to return something', 'Cancel my order please', 'Goodbye!'
]
print('Intent Classification Results:')
for msg in test_msgs:
    pred = intent_clf.predict([msg])[0]
    prob = max(intent_clf.predict_proba([msg])[0])
    print(f'  "{msg}" -> {pred} ({prob:.2f})')

Intent Classification Results:
  "Hi there!" -> greeting (0.34)
  "When will my package arrive?" -> check_order (0.32)
  "I need to return something" -> return_item (0.27)
  "Cancel my order please" -> cancel_order (0.31)
  "Goodbye!" -> farewell (0.40)


In [3]:
import re

class SlotFiller:
    """Extract key information from user utterances."""
    PATTERNS = {
        'order_id': r'order\s*(?:number|id|#)?\s*([A-Z0-9]{6,12})',
        'date': r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2})',
        'quantity': r'(\d+)\s+(?:item|unit|piece|box)s?',
    }
    def extract(self, text):
        return {slot: m.group(1).strip()
                for slot, pat in self.PATTERNS.items()
                if (m := re.search(pat, text, re.IGNORECASE))}

filler = SlotFiller()
for u in [
    'I want to return 2 items from order ORD123456',
    'Please cancel order ABC789012 placed on Jan 15',
    'How do I track order XYZ999888?'
]:
    slots = filler.extract(u)
    print(f'Input:  {u}')
    print(f'Slots:  {slots}')
    print()

Input:  I want to return 2 items from order ORD123456
Slots:  {'order_id': 'ORD123456', 'quantity': '2'}

Input:  Please cancel order ABC789012 placed on Jan 15
Slots:  {'order_id': 'ABC789012', 'date': 'Jan 15'}

Input:  How do I track order XYZ999888?
Slots:  {'order_id': 'XYZ999888'}



In [4]:
class DialogueStateTracker:
    """Track conversation state across multiple turns."""
    def __init__(self, clf, filler):
        self.clf = clf
        self.filler = filler
        self.state = {'intents': [], 'slots': {}, 'turn': 0}

    def update(self, utterance):
        self.state['turn'] += 1
        intent = self.clf.predict([utterance])[0]
        self.state['intents'].append(intent)
        self.state['slots'].update(self.filler.extract(utterance))
        return intent

    def respond(self, intent):
        slots = self.state['slots']
        oid = slots.get('order_id', '(need order ID)')
        responses = {
            'greeting': 'Hello! How can I help you today?',
            'check_order': f'Checking order {oid}...' if 'order_id' in slots else 'Please provide your order number.',
            'cancel_order': f'Cancelling order {oid}.' if 'order_id' in slots else 'What is the order number to cancel?',
            'return_item': 'I can help with returns. Please share your order number.',
            'farewell': 'Thank you for contacting us! Goodbye!',
        }
        return responses.get(intent, "I'm not sure how to help with that.")

tracker = DialogueStateTracker(intent_clf, filler)
convo = ['Hi there!', 'I want to cancel my order', 'The order number is ORD789012', 'Great, thanks! Goodbye']
print('Multi-turn Conversation:')
for turn, msg in enumerate(convo, 1):
    intent = tracker.update(msg)
    response = tracker.respond(intent)
    print(f'Turn {turn}')
    print(f'  User:   {msg}')
    print(f'  Intent: {intent}  Slots: {tracker.state["slots"]}')
    print(f'  Bot:    {response}')
    print()

Multi-turn Conversation:
Turn 1
  User:   Hi there!
  Intent: greeting  Slots: {}
  Bot:    Hello! How can I help you today?

Turn 2
  User:   I want to cancel my order
  Intent: cancel_order  Slots: {}
  Bot:    What is the order number to cancel?

Turn 3
  User:   The order number is ORD789012
  Intent: check_order  Slots: {'order_id': 'number'}
  Bot:    Checking order number...

Turn 4
  User:   Great, thanks! Goodbye
  Intent: farewell  Slots: {'order_id': 'number'}
  Bot:    Thank you for contacting us! Goodbye!



In [5]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

class RetrievalChatbot:
    """FAQ chatbot using TF-IDF cosine similarity."""
    FAQ = [
        ('How do I reset my password?',
         'Go to Settings > Account > Reset Password, then enter your email.'),
        ('What payment methods do you accept?',
         'We accept Visa, MasterCard, PayPal, and Apple Pay.'),
        ('How long does shipping take?',
         'Standard: 3-5 business days. Express: 1-2 days.'),
        ('Can I return a product?',
         'Yes! Free returns within 30 days of purchase.'),
        ('How do I track my order?',
         'Visit Orders in your account and click Track Shipment.'),
        ('Do you offer student discounts?',
         'Yes, 20% off for verified students via our EDU program.'),
    ]
    def __init__(self):
        questions, self.answers = zip(*self.FAQ)
        self.vec = TfidfVectorizer(ngram_range=(1, 2))
        self.faq_vecs = self.vec.fit_transform(questions)

    def respond(self, query, threshold=0.2):
        qv = self.vec.transform([query])
        sims = cosine_similarity(qv, self.faq_vecs).flatten()
        idx = sims.argmax()
        if sims[idx] >= threshold:
            return self.answers[idx], sims[idx]
        return 'Please contact support@example.com for help.', 0.0

rcbot = RetrievalChatbot()
for q in [
    'How can I change my password?',
    'What credit cards do you take?',
    'When will my package arrive?',
    'Is there a discount for college students?',
    'What is the meaning of life?',
]:
    ans, score = rcbot.respond(q)
    print(f'Q: {q}')
    print(f'A: {ans}  (sim={score:.3f})')
    print()

Q: How can I change my password?
A: Go to Settings > Account > Reset Password, then enter your email.  (sim=0.577)

Q: What credit cards do you take?
A: We accept Visa, MasterCard, PayPal, and Apple Pay.  (sim=0.450)

Q: When will my package arrive?
A: Go to Settings > Account > Reset Password, then enter your email.  (sim=0.306)

Q: Is there a discount for college students?
A: Please contact support@example.com for help.  (sim=0.000)

Q: What is the meaning of life?
A: We accept Visa, MasterCard, PayPal, and Apple Pay.  (sim=0.321)



In [6]:
# DialoGPT: generative open-domain chatbot
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('Loading DialoGPT-small...')
tokenizer = AutoTokenizer.from_pretrained('microsoft/DialoGPT-small')
model = AutoModelForCausalLM.from_pretrained('microsoft/DialoGPT-small')

def chat(history, user_input):
    new_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    bot_input = torch.cat([history, new_ids], dim=-1) if history is not None else new_ids
    if bot_input.shape[-1] > 800:
        bot_input = bot_input[:, -800:]
    response_ids = model.generate(
        bot_input, max_length=bot_input.shape[-1] + 100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True, temperature=0.7, top_p=0.9,
    )
    text = tokenizer.decode(response_ids[:, bot_input.shape[-1]:][0], skip_special_tokens=True)
    return text, response_ids

history = None
for msg in ['Hello! How are you?', 'Tell me about AI.', 'That sounds interesting!']:
    print(f'User: {msg}')
    reply, history = chat(history, msg)
    print(f'Bot:  {reply}')
    print()

/opt/miniconda3/envs/nlp-course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading DialoGPT-small...
User: Hello! How are you?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Bot:  Hello, neighbor!

User: Tell me about AI.
Bot:  AI. It's a real game.

User: That sounds interesting!
Bot:  Maybe AI games will be a thing.



## Practice Exercises

See **`Lecture-15-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Type | Method | Best For |
|------|--------|----------|
| Rule-based | Keyword matching | Narrow domain, predictable input |
| Retrieval | TF-IDF cosine similarity | FAQ chatbots |
| Intent+Slot | ML classifier + regex | Task-oriented bots |
| Generative | DialoGPT, GPT | Open-domain, creative |

**Next Lecture**: NLP Applications & Case Studies.

---
*Book reference: Practical NLP Ch.9*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**